# Manipulation Simulation Experiments

This notebook is your lab bench for running experiments, computing metrics, and generating plots.

## Experiment Structure

1. **Imports + Config**: Load the simulation package and configure parameters
2. **Run Simulations**: Execute many episodes across different conditions
3. **Compute Metrics**: Extract and calculate manipulation metrics
4. **Plots + Tables**: Visualize results and create paper-ready figures
5. **Save Summary**: Export results for later analysis

## Section 1: Imports + Config

In [ ]:
import json
import glob
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

# Import manipulation simulation modules
from manipulation_sim.simulate import run_episode, save_run, load_run
from manipulation_sim.metrics import (
    exploitation,
    credit_capture,
    risk_externalization,
    manipulation_index,
    compute_ate,
    compute_ate_by_agent,
)
from manipulation_sim.analysis import (
    load_all_runs,
    runs_to_dataframe,
    plot_utility_vs_labor,
    plot_metrics_comparison,
    plot_ate_bars,
    print_summary_table,
    compute_and_print_ate,
)
from manipulation_sim import config

# Configure plotting
sns.set_style("whitegrid")
plt.rcParams["figure.figsize"] = (10, 6)
plt.rcParams["font.size"] = 11

print("Imports complete!")
print(f"Config: {config.N_AGENTS} agents, {config.N_DAYS} days")

## Section 2: Run Simulations

Run many episodes across different conditions:
- **Aligned** vs **Misaligned** incentives
- **Communication** enabled vs disabled
- Multiple random seeds for statistical power

In [ ]:
# Configuration (as specified in the spec)
N_SEEDS = 50  # Number of random seeds for each condition
N_DAYS = config.N_DAYS
LOG_DIR = "manipulation_sim/logs"

# Ensure log directory exists
Path(LOG_DIR).mkdir(parents=True, exist_ok=True)

results = []

print(f"Running {N_SEEDS} episodes per condition...")
print("This may take a moment...\n")

# Run simulations (matching spec structure exactly)
for aligned in [False, True]:
    for seed in range(N_SEEDS):
        run_id = f"run_aligned{int(aligned)}_{seed}"
        
        # Run episode
        log, rewards = run_episode(
            run_id=run_id,
            aligned=aligned,
            n_days=N_DAYS,
            rng_seed=seed,
            allow_communication=True,  # Communication enabled by default
        )
        
        # Extract simple labor/credit/risk from final state (as per spec)
        final_day = log[-1]
        world_state = final_day["world"]
        
        labor = {name: data["labor"] for name, data in world_state["agents"].items()}
        credit = {name: data["credit"] for name, data in world_state["agents"].items()}
        risk_actions = {name: data["risk_actions"] for name, data in world_state["agents"].items()}
        utility = rewards
        
        # Compute metrics (as per spec)
        E = exploitation(labor, utility)
        CC = credit_capture(credit, labor)
        RE = risk_externalization(risk_actions, utility)
        
        # Store per-agent (as per spec)
        for name in labor.keys():
            results.append({
                "aligned": aligned,
                "seed": seed,
                "agent": name,
                "E": E[name],
                "CC": CC[name],
                "RE": RE[name],
                "labor": labor[name],
                "utility": utility[name],
                "credit": credit[name],
                "risk": risk_actions[name],
            })
        
        # Save run for later replay
        metadata = {
            "aligned": aligned,
            "n_days": N_DAYS,
            "seed": seed,
        }
        save_run(run_id, log, rewards, metadata=metadata, path=LOG_DIR)

print(f"Completed {len(results)} agent-run combinations!")
print(f"Saved logs to {LOG_DIR}/")

# Convert to DataFrame (as per spec)
df = pd.DataFrame(results)
print(f"\nDataFrame shape: {df.shape}")
print(df.head())

## Section 3: Compute Metrics

Calculate manipulation metrics (E_i, CC_i, RE_i, M_i) for each run.

In [ ]:
# Metrics are already computed in Section 2
# df is already created from results

print("DataFrame shape:", df.shape)
print("\nFirst few rows:")
print(df.head())
print("\nSummary statistics:")
print(df.describe())

# Optional: Load saved runs for additional analysis
# all_runs = load_all_runs(f"{LOG_DIR}/*.json")
# df_full = runs_to_dataframe(all_runs)

## Section 4: Plots + Tables

Generate paper-ready visualizations and summary tables.

In [ ]:
# Plot 1: Utility vs Labor scatter (as per spec)
# Separate plots for aligned/misaligned conditions

for aligned in [False, True]:
    sub = df[df["aligned"] == aligned]
    plt.figure(figsize=(10, 6))
    for agent in sub["agent"].unique():
        agent_data = sub[sub["agent"] == agent]
        plt.scatter(agent_data["labor"], agent_data["utility"], label=agent, alpha=0.6, s=50)
    plt.title(f"Utility vs Labor (aligned={aligned})")
    plt.xlabel("Labor")
    plt.ylabel("Utility")
    plt.legend()
    plt.grid(alpha=0.3)
    plt.show()

In [ ]:
# Plot 2: Metrics comparison across conditions (optional - using analysis helpers)
# These plots use the analysis module functions for convenience

# Load all runs for full analysis
all_runs = load_all_runs(f"{LOG_DIR}/*.json")
df_full = runs_to_dataframe(all_runs)

if len(df_full) > 0:
    plot_metrics_comparison(df_full, metric="exploitation", save_path="exploitation_comparison.png")
    plot_metrics_comparison(df_full, metric="credit_capture", save_path="credit_capture_comparison.png")
    plot_metrics_comparison(df_full, metric="risk_externalization", save_path="risk_externalization_comparison.png")
else:
    print("No saved runs found. Run Section 2 first.")

In [ ]:
# Print summary tables (as per spec)
# Compare misaligned vs aligned

df_group = df.groupby(["aligned", "agent"])[["E", "CC", "RE"]].mean().reset_index()
print("\n=== Mean Metrics by Condition and Agent ===")
print(df_group)

# Also show utility vs labor comparison
df_group_util = df.groupby(["aligned", "agent"])[["labor", "utility"]].mean().reset_index()
print("\n=== Mean Labor and Utility by Condition and Agent ===")
print(df_group_util)

In [ ]:
# Compute and print ATE results (Average Treatment Effect)
# ATE = E[Y | T=1] - E[Y | T=0] where T=1 is misaligned, T=0 is aligned

print("=== ATE Analysis: Exploitation (E) ===")
for agent in sorted(df["agent"].unique()):
    aligned_values = df[(df["aligned"] == True) & (df["agent"] == agent)]["E"].tolist()
    misaligned_values = df[(df["aligned"] == False) & (df["agent"] == agent)]["E"].tolist()
    
    if aligned_values and misaligned_values:
        ate_result = compute_ate(misaligned_values, aligned_values)
        print(f"\n{agent}:")
        print(f"  ATE: {ate_result['ate']:.4f}")
        print(f"  95% CI: [{ate_result['ci_lower']:.4f}, {ate_result['ci_upper']:.4f}]")
        print(f"  Misaligned mean: {ate_result['treatment_mean']:.4f}")
        print(f"  Aligned mean: {ate_result['control_mean']:.4f}")

print("\n=== ATE Analysis: Credit Capture (CC) ===")
for agent in sorted(df["agent"].unique()):
    aligned_values = df[(df["aligned"] == True) & (df["agent"] == agent)]["CC"].tolist()
    misaligned_values = df[(df["aligned"] == False) & (df["agent"] == agent)]["CC"].tolist()
    
    if aligned_values and misaligned_values:
        ate_result = compute_ate(misaligned_values, aligned_values)
        print(f"{agent}: ATE = {ate_result['ate']:.4f} [{ate_result['ci_lower']:.4f}, {ate_result['ci_upper']:.4f}]")

In [ ]:
# Plot ATE with confidence intervals (as per spec)
# Visualize the causal effect of misaligned vs aligned incentives

ate_by_agent = {}
for agent in sorted(df["agent"].unique()):
    aligned_values = df[(df["aligned"] == True) & (df["agent"] == agent)]["E"].tolist()
    misaligned_values = df[(df["aligned"] == False) & (df["agent"] == agent)]["E"].tolist()
    
    if aligned_values and misaligned_values:
        ate_by_agent[agent] = compute_ate(misaligned_values, aligned_values)

if ate_by_agent:
    plot_ate_bars(ate_by_agent, metric_name="Exploitation (E)", save_path="ate_exploitation.png")
else:
    print("No data available for ATE plotting.")

## Section 5: Save Summary for Paper

Export results to CSV for further analysis or inclusion in papers.

In [ ]:
# Save summary for paper (as per spec)
# Export results to CSV for further analysis or inclusion in papers

df.to_csv("results_full.csv", index=False)
print("Saved full results to results_full.csv")

# Create aggregated summary (as per spec)
summary = df.groupby(["aligned", "agent"])[["E", "CC", "RE", "labor", "utility"]].agg({
    "E": ["mean", "std"],
    "CC": ["mean", "std"],
    "RE": ["mean", "std"],
    "labor": ["mean", "std"],
    "utility": ["mean", "std"],
}).round(3)

summary.to_csv("results_summary.csv")
print("Saved aggregated summary to results_summary.csv")
print("\nSummary Table (paper-ready):")
print(summary)